<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0">

# Derivatives Portfolio Risk Statistics

From a risk management perspective it is important to know **how sensitive derivatives portfolios are** with regard to certain parameter values (market quotes, model assumptions, etc.). This part illustrates how to generate certain **risk reports** for `derivatives_portfolio` objects.

In [ ]:
!git clone https://github.com/tpq-classes/computational_finance_dx.git
import sys
sys.path.append('computational_finance_dx')

!git clone https://github.com/yhilpisch/dx.git
sys.path.append('dx')


In [ ]:
import dx
import time
import numpy as np
import datetime as dt

## Risk Factors

The example is based on **two risk factors**, both modeled as geometric Brownian motions.

In [ ]:
# constant short rate
r = dx.constant_short_rate('r', 0.01)

In [ ]:
# market environment
me_gbm_1 = dx.market_environment('gbm_1', dt.datetime(2020, 1, 1))

In [ ]:
# geometric Brownian motion
me_gbm_1.add_constant('initial_value', 40.)
me_gbm_1.add_constant('volatility', 0.2)
me_gbm_1.add_constant('currency', 'EUR')
me_gbm_1.add_constant('model', 'gbm')

In [ ]:
me_gbm_2 = dx.market_environment('gbm_2', me_gbm_1.pricing_date)

In [ ]:
# valuation environment
val_env = dx.market_environment('val_env', dt.datetime(2020, 1, 1))
val_env.add_constant('paths', 75000)
    # 75,000 paths
val_env.add_constant('frequency', 'W')
    # weekly frequency
val_env.add_curve('discount_curve', r)
val_env.add_constant('starting_date', dt.datetime(2020, 1, 1))
val_env.add_constant('final_date', dt.datetime(2020, 12, 31))

In [ ]:
# add valuation environment to market environments
me_gbm_1.add_environment(val_env)
me_gbm_2.add_environment(me_gbm_1)
me_gbm_2.add_constant('initial_value', 40.)
me_gbm_2.add_constant('volatility', 0.5)
  # higher volatility

In [ ]:
risk_factors = {'gbm_1' : me_gbm_1, 'gbm_2' : me_gbm_2}
  # market with two risk factors

## Derivatives Positions

We are going to model **total of 6 derivatives positions**.

### Market Environment

All derivatives instruments (positions) share the same `market_environment` object.

In [ ]:
# market environment for the options
me_option = dx.market_environment('put', dt.datetime(2020, 1, 1))
me_option.add_constant('maturity', dt.datetime(2020, 12, 31))
me_option.add_constant('currency', 'EUR')
me_option.add_environment(val_env)

### Derivatives Positions

Two different kinds of derivatives make up the portfolio---an **American put option** and a **European maximum call option**. Both types of derivatives populate three positions, respectively.

In [ ]:
positions = {}
half = 3  # 2 times that many options
for i in range(half):
    name = 'am_put_pos_%s' %i  # same name for position key and name
    positions[name] = dx.derivatives_position(
                        name=name,
                        quantity=1,
                        underlyings=['gbm_1'],
                        mar_env=me_option,
                        otype='American single',
                        payoff_func='np.maximum(instrument_values - 40., 0)')

multi_payoff = "np.maximum(np.maximum(maturity_value['gbm_1'], maturity_value['gbm_2']) - 40., 0)"
for i in range(half, 2 * half):
    name = 'multi_pos_%s' %i  # same name for position key and name
    positions[name] = dx.derivatives_position(
                        name=name,
                        quantity=1,
                        underlyings=['gbm_1', 'gbm_2'],
                        mar_env=me_option,
                        otype='European multi',
                        payoff_func=multi_payoff)

## Portfolio Modeling and Valuation

The instantiation of the `derivatives_portfolio` object is as usual.

In [ ]:
portfolio = dx.derivatives_portfolio(
                        name='portfolio',
                        positions=positions,
                        val_env=val_env,
                        risk_factors=risk_factors,
                        correlations=None,
                        parallel=False)

In [ ]:
%time res = portfolio.get_values(fixed_seed=True)

Here, the **value estimates** from the Monte Carlo simulation and valuation.

In [ ]:
res

## Portfolio Risk Reports

Portfolio risk reports are meant to provide a broad overview of how sensitive the value of a portfolio is with regard to the value of certain input parameters (market data, model parameters). While **Greeks** provide the same information with regard to marginal changes in the input paramters, risk reports provide a **wider range input-output (parameter-portfolio value) combinations**.

### No Correlation

First, consider the portfolio from before, i.e. **without correlation**.

In [ ]:
portfolio.val_env.get_list('cholesky_matrix')

Calling the method `get_port_risk` and providing a key for the respetive Greek yields sensitivities with regard to all risk factors (here: `gbm_1` and `gbm_2`).

In [ ]:
%%time
vegas, benchvalue = portfolio.get_port_risk(Greek='Vega',
                                fixed_seed=True)

The return object is a pandas `Panel` object.

In [ ]:
vegas

Using the helper funtion `risk_report` allows the easy, readable printout of the results, i.e. the **portfolio volatility sensitivities**. In this case you can see that, for example, the increase in the first risk fator's (`gbm_1`) volatility by 10% leads to a portfolio value increase bya bit less than 1 currency unit. Decreasing the same input parameter by 10% reduces the portfolio value by a bit less than 1 currency unit.

In [ ]:
dx.risk_report(vegas)

Of course, you can generate the same risk report for the **portfolio initial value sensitivities**.

In [ ]:
%time deltas, benchvalue = portfolio.get_port_risk(Greek='Delta', fixed_seed=True)

For example, increasing the initial value of the first risk factor (`gbm_1`) by 10% increases the portfolio value by about 11 currency units.

In [ ]:
deltas

In [ ]:
deltas.loc(axis=0)[:, 'value'] - benchvalue

### With Correlation

Consider now a **highly negative correlation** case.

In [ ]:
correlations = [['gbm_1', 'gbm_2', -0.9]]

In [ ]:
portfolio = dx.derivatives_portfolio(
                        'portfolio', positions, val_env,
                        risk_factors, correlations, parallel=False)

In [ ]:
portfolio.val_env.get_list('cholesky_matrix')

Since the value of the European maximum call option is dependent on the risk factor correlation you see a **significant change in this derivative's value estimate**.

In [ ]:
%time portfolio.get_values(fixed_seed=True)

Via the `step` parameter, you can influence the  **granularity of the risk report**.

In [ ]:
%%time
deltas, benchvalue = portfolio.get_port_risk(Greek='Delta',
                                 fixed_seed=True,
                                 step=0.05)

In this case, an increase in the intial value of the first risk factor (`gbm_1`) by 10% leads to a **much higher increase**
in the portfolio value of about 15 currency units.

In [ ]:
deltas

In [ ]:
deltas.loc(axis=0)[:, 'value'] - benchvalue

**Copyright, License & Disclaimer**

© Dr. Yves J. Hilpisch | The Python Quants GmbH

DX Analytics (the "dx library" or "dx package") is licensed under the GNU Affero General
Public License version 3 or later (see http://www.gnu.org/licenses/).

DX Analytics comes with no representations or warranties, to the extent
permitted by applicable law.

http://tpq.io | [dx@tpq.io](mailto:team@tpq.io) |
http://twitter.com/dyjh

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

**Quant Platform** | http://pqp.io

**Python for Finance Training** | http://training.tpq.io

**Certificate in Computational Finance** | http://compfinance.tpq.io

**Derivatives Analytics with Python (Wiley Finance)** |
http://dawp.tpq.io

**Python for Finance (2nd ed., O'Reilly)** |
http://py4fi.tpq.io